# INP-toolkit tutorial

Start with droplet counts, assign a water blank, and calculate an ice-nucleating
particle (INP) concentration curve. These are **invented example data**, not a
research dataset. We measure one sample at two dilutions in one freezing cycle.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

import inptk

## 1. Counts and sample information

Each row describes one set of wells at one temperature. `n_frozen` is the number
frozen so far; `n_total` is the number of wells being observed. The blank has its
own wells. Repeated cycles must have separate `cycle_id` values.

In [ ]:
temperatures = [-6, -8, -10, -12, -14, -16, -18]
counts = pd.concat([
    pd.DataFrame({
        "measurement_id": name,
        "cycle_id": "1",
        "temperature_C": temperatures,
        "n_frozen": frozen,
        "n_total": total,
    })
    for name, frozen, total in [
        ("neat", [0, 1, 4, 10, 19, 28, 32], 32),
        ("diluted", [0, 0, 0, 1, 3, 7, 15], 32),
        ("blank", [0, 0, 0, 0, 1, 2, 3], 16),
    ]
], ignore_index=True)
counts.head()

The dilution factor is 1 for the undiluted input and 10 for the diluted input.
All droplets here are 50 µL. The example sample represents 1,000 litres of air
collected into 10 mL of suspension, using the whole filter. Those volumes are
needed only when converting the result to INP per litre of air.

In [ ]:
metadata = pd.DataFrame([
    {"measurement_id": "neat", "sample_id": "sample", "dilution": 1,
     "droplet_volume_uL": 50, "sample_type": "air", "air_volume_L": 1000,
     "suspension_volume_mL": 10, "filter_fraction_used": 1},
    {"measurement_id": "diluted", "sample_id": "sample", "dilution": 10,
     "droplet_volume_uL": 50, "sample_type": "air", "air_volume_L": 1000,
     "suspension_volume_mL": 10, "filter_fraction_used": 1},
    {"measurement_id": "blank", "sample_id": "water", "dilution": 1,
     "droplet_volume_uL": 50, "sample_type": "other"},
])
experiment = inptk.read_counts(
    counts,
    metadata=metadata,
    water_blank_map={"neat": ["blank"], "diluted": ["blank"]},
)

Blank assignment is explicit: names alone never select a blank. The 16 blank
wells are kept separate from the 32 sample wells.

For an Icescopy export, use `inptk.read_icescopy("freeze_count_timeseries.csv", ...)`
instead of `read_counts()`. It also accepts an `.icescopy` project and reads its
available physical metadata. Supply `sample_map` and `water_blank_map` yourself;
use `metadata` to fill in any missing volumes.

## 2. Calculate the concentration curve

Choose a name for the output curve and list the inputs and cycle it uses.
`mle` means maximum likelihood estimation: it fits the sample and blank counts
together, including uncertainty in the blank. It does not subtract a fixed blank
curve first. Concentration is constrained to stay constant or increase as the
sample cools.

Here we select counts on a 0.5 °C grid before fitting. The default temperature
rule uses the latest observation at or warmer than each grid temperature.
The grid uses the observed endpoints unless you specify them.

In [ ]:
curves = {"sample": {"inputs": ["neat", "diluted"], "cycle": "1"}}
result = inptk.analyze_concentration(
    experiment,
    curves=curves,
    method="mle",
    temperature_step_C=0.5,
    output_basis="sampled_air",
)
spectrum = result.curves["sample"].cumulative.to_dataframe()
spectrum[["temperature_C", "concentration", "lower_error", "upper_error", "unit"]].head()

`concentration` is cumulative: it includes particles active at that temperature
or warmer. `lower_error` and `upper_error` are distances from the estimate to the
uncertainty limits, not the limits themselves.

The reported curve includes only temperatures from the first through the last
observed sample freezing event. Earlier and later observations still constrain
the fit. Blank-corrected zeros inside that interval are kept.

Individual curves use their original freezing interval. Ranges chosen for the
combined curve do not remove individual points.


In [ ]:
plot = spectrum.sort_values("temperature_C")
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(plot.temperature_C, plot.concentration, "o-", label="Sample")
ax.fill_between(
    plot.temperature_C,
    plot.concentration - plot.lower_error,
    plot.concentration + plot.upper_error,
    alpha=0.2,
    label="Nominal 95% interval",
)
ax.set(xlabel="Temperature (°C)", ylabel="INP per litre of air")
ax.legend()
plt.show()

## 3. The same calculation, step by step

Use this form when you want to inspect an intermediate table or stop before
converting units. The first concentration table is in suspension units;
`convert_concentration()` converts it to air units. `finalize_spectrum()` applies
the final point-selection rule. MLE already constrains the fitted curve to be
monotonic, so this step does not impose a second fit.

In [ ]:
fractions = inptk.frozen_fraction(experiment)
estimated = inptk.estimate_concentration(
    fractions,
    experiment=experiment,
    curves=curves,
    method="mle",
    temperature_step_C=0.5,
)
converted = inptk.convert_concentration(
    estimated, experiment.samples, basis="sampled_air",
)
final = inptk.finalize_spectrum(converted)
final.to_dataframe().head()

## 4. Average instead of fitting

Average combines concentration estimates from the eligible inputs at each
temperature. Its default automatic limits select a continuous, nondecreasing
curve during cooling, using the earlier input as far as possible before moving
to the next. You can inspect the proposed limits before calculating.

In [ ]:
limits = inptk.suggest_temperature_ranges(
    experiment, curves=curves, temperature_step_C=0.5,
)
limits.inputs

In [ ]:
average = inptk.analyze_concentration(
    experiment,
    curves=curves,
    method="average",
    temperature_ranges_C=limits.temperature_ranges_C,
    temperature_step_C=0.5,
    output_basis="sampled_air",
)
average.curves["sample"].cumulative.to_dataframe().head()

## Saving and using your own data

Save the complete analysis with `result.save("analysis.inptk")` and reopen it
with `inptk.load("analysis.inptk")`. To export just the concentration table, use
`spectrum.to_csv("concentration.csv", index=False)`.

Replace the count arrays and metadata above with your own inputs. Use a separate
named curve for each sample or repeated cycle; combining cycles would count
repeated measurements of the same wells as if they were new wells.